# Cross-shelf wave forecasting and extreme-value analysis: Colab pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/[USER]/cross-shelf-wave-hs-bay-of-bengal/blob/main/notebooks/Cross_shelf_wave_pipeline.ipynb)

This notebook runs the complete analysis of the repository on Google Colab, with all inputs and outputs stored on Google Drive:
baseline and LSTM wave forecasts, storm-case evaluation, extreme-value analysis, depth diagnostics, tables and figures.

**Inputs** (on Google Drive)
- one parquet file per station (`P1_merged_raw.parquet` ... `P5_merged_raw.parquet`) with the columns
  `time, u10, v10, sst, VHM0, VTPK` (ERA5 wind and sea-surface temperature, CMEMS significant wave height and peak period);
  alternatively, build the station series from NetCDF downloads with `scripts/download_data.py` and `scripts/build_station_series.py`;
- optional: the NetCDF source files (to report the grid cells actually used) and the altimeter mission files.

**Outputs** are written to `<PROJECT_DIR>/outputs/`. The footprint is well below 1 GB for the full set of 375 LSTM runs
(predictions are stored gzip-compressed).

| Step | Description | Runtime | Approximate time |
|---|---|---|---|
| 1-3 | Setup, code, unit tests | CPU | 5 min |
| 4-5 | Station series and data audit | CPU | 5 min |
| 6 | Baselines (persistence, ridge, XGBoost) | CPU | 15-30 min |
| 7-8 | Cyclone tracks, storm windows, extreme-value analysis, depth diagnostics | CPU | 15-30 min |
| 9 | LSTM training (resumable) | GPU (T4 or better) | see the timing estimate in step 9 |
| 10 | Storm evaluation, aggregated tables, figures | CPU | 10 min |
| 11 | Optional: altimetry match-up | CPU | 10 min |
| 12 | Archive the results | CPU | 5 min |

All steps write their outputs to Drive and skip work that already exists, so a disconnected session can be resumed by
re-running steps 1-2 and continuing from the step that was interrupted.

## 1. Setup: mount Drive and configure paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib, shutil, subprocess, zipfile
import pandas as pd

DRIVE = pathlib.Path('/content/drive/MyDrive')

# --- Configuration: adjust to your Drive layout --------------------------------------------------------------
PROJECT_DIR = DRIVE / 'wave_paper'                                      # project folder (code archive and outputs)
OUT_DIR = PROJECT_DIR / 'outputs'                                       # all results are written here
INPUT_PARQUET_DIR = DRIVE / 'wave_thesis' / 'data' / 'processed'        # P1_merged_raw.parquet ... P5_merged_raw.parquet
WAVE_NC = DRIVE / 'wave_cmems_nbob.nc'                                  # optional: wave reanalysis NetCDF (grid-cell report)
WIND_NC = DRIVE / 'era5_yearly_backup' / 'wind_nbob_1993.nc'            # optional: one ERA5 wind NetCDF (grid-cell report)
ALT_DIR = DRIVE / 'wave_thesis' / 'data' / 'altimetry' / 'altimetry_swh_nbob_1993_2023_(2)'   # optional: altimeter mission files
# -------------------------------------------------------------------------------------------------------------

OUT_DIR.mkdir(parents=True, exist_ok=True)
for name, path in [('input parquet files', INPUT_PARQUET_DIR), ('wave NetCDF (optional)', WAVE_NC),
                   ('wind NetCDF (optional)', WIND_NC), ('altimeter files (optional)', ALT_DIR)]:
    print(f"{'found  ' if path.exists() else 'MISSING'}  {name}: {path}")

The input parquet files are required for step 4. The NetCDF files are only used to report the grid cells that were
actually extracted (step 5), and the altimeter files only for the optional step 11; the pipeline runs without them.

## 2. Get the code and install dependencies

In [ ]:
REPO_URL = 'https://github.com/[USER]/cross-shelf-wave-hs-bay-of-bengal.git'
WORK_DIR = pathlib.Path('/content/work'); WORK_DIR.mkdir(exist_ok=True)
REPO_DIR = WORK_DIR / 'cross-shelf-wave-hs-bay-of-bengal'
CODE_ZIP = PROJECT_DIR / 'cross-shelf-wave-hs-bay-of-bengal.zip'

if CODE_ZIP.exists():                      # use an archived release stored on Drive, if present
    with zipfile.ZipFile(CODE_ZIP) as z:
        z.extractall(WORK_DIR)
elif not REPO_DIR.exists():                # otherwise clone the repository
    !git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)

!pip -q install pytest xarray netCDF4 pyarrow
STATION_DIR = str(OUT_DIR / 'data' / 'stations')    # station CSV files
RESULTS_DIR = str(OUT_DIR)                          # root of all result folders
print('working directory:', os.getcwd())

## 3. Run the unit tests

In [ ]:
!python -m pytest -q

Expected result: all tests pass (the PyTorch tests are skipped when PyTorch is not installed).

## 4. Build the station series

Converts the per-station parquet files into the CSV layout used by all scripts (3-hourly series with columns `time, u10, v10, sst_k, hs, tp`).

In [ ]:
!python scripts/convert_station_parquet.py --src "{INPUT_PARQUET_DIR}" --out "{STATION_DIR}"

Expected result: one line per station reporting the date range (1993-01-01 to 2023-12-31), `missing steps 0` and `NaN values 0`.
Any missing time step must be resolved before continuing, because the model inputs require a regular 3-hourly grid.

## 5. Data audit

Reports the continuity of each series and descriptive statistics (mean, 90th and 99th percentile and maximum of Hs, mean Tp, wind speed and
direction). If the NetCDF files are available, it also reports the wave-model and ERA5 grid cells that correspond to each station.

In [ ]:
EXTRA = f'--wave-nc "{WAVE_NC}" --wind-nc "{WIND_NC}"' if WAVE_NC.exists() and WIND_NC.exists() else ''
!python scripts/audit_stations.py --data "{STATION_DIR}" --out "{RESULTS_DIR}/tables" {EXTRA}

Outputs: `tables/station_descriptive_stats.csv` and, when the NetCDF files are given, `tables/grid_cells.csv`.
Station depths in `config/stations.yaml` come from the GEBCO grid; update them there if the station positions change.

## 6. Baseline forecasts

Persistence, ridge regression and gradient-boosted trees (XGBoost) for every station and lead time (6, 12 and 24 h).
XGBoost dominates the run time (15-30 min on a CPU runtime); add `--skip-xgb` to omit it.

In [ ]:
!python scripts/run_baselines.py --data "{STATION_DIR}" --out "{RESULTS_DIR}/baselines"

Expected result: one line per station and lead time (15 in total). Output: `baselines/baseline_metrics.csv` and the test-set predictions.

## 7. Cyclone tracks and storm windows

Downloads the North Indian Ocean subset of IBTrACS (NOAA NCEI) and lists the storms that came within a given distance of the stations.
If the download fails, obtain `ibtracs.NI.list.v04r01.csv` from the NCEI website and pass its path with `--csv`.

In [ ]:
!python scripts/make_ibtracs_subset.py --out "{RESULTS_DIR}/ibtracs_nio.csv"
!python scripts/list_storms.py --tracks "{RESULTS_DIR}/ibtracs_nio.csv" --from 2019 --to 2023 --max-km 800 --out "{RESULTS_DIR}/tables/storms_listed.csv"

The table lists the storm name, dates and the closest approach (km) to each station. Define the storm windows for the storm-case evaluation
(start and end date) from this list; they are written to `config/params.yaml`.

In [ ]:
import yaml

STORMS = {   # name: [start, end]
    'Fani':    ['2019-04-26', '2019-05-04'],
    'Bulbul':  ['2019-11-05', '2019-11-11'],
    'Amphan':  ['2020-05-18', '2020-05-22'],
    'Yaas':    ['2021-05-23', '2021-05-27'],
    'Sitrang': ['2022-10-22', '2022-10-26'],
    'Mocha':   ['2023-05-12', '2023-05-16'],
    'Midhili': ['2023-11-15', '2023-11-18'],
}
cfg_path = REPO_DIR / 'config' / 'params.yaml'
cfg = yaml.safe_load(open(cfg_path))
cfg['storms_in_test_period'] = STORMS
yaml.safe_dump(cfg, open(cfg_path, 'w'), sort_keys=False)
print('storm windows written:', list(STORMS))

## 8. Extreme-value analysis and depth diagnostics

Peaks-over-threshold analysis for each station: threshold diagnostics, sensitivity to threshold and declustering window, year-bootstrap and
profile-likelihood intervals, separate fits for cyclone-related and other storms, annual-maxima GEV, and trend and change-point tests.
The cyclone distance limit is varied (500, 800 and 1000 km) to quantify its effect.

In [ ]:
!python scripts/run_evt.py --data "{STATION_DIR}" --out "{RESULTS_DIR}/evt" --tracks "{RESULTS_DIR}/ibtracs_nio.csv"
for km in (500, 1000):
    subprocess.run(f'python scripts/run_evt.py --data "{STATION_DIR}" --out "{RESULTS_DIR}/evt_km{km}" --tracks "{RESULTS_DIR}/ibtracs_nio.csv" --cyclone-km {km}', shell=True, check=True)
pd.read_csv(f'{RESULTS_DIR}/evt/evt_summary.csv').set_index('station').T

How to read `evt_summary.csv`:
- `xi`, `xi_lo`, `xi_hi`: GPD shape parameter with its 95% profile-likelihood interval; an interval that contains zero does not support a bounded tail.
- `rl100` (pooled POT), `rl100_combined` (cyclone-related and other storms treated separately) and `gev_rl100` (annual maxima) are three estimates of the 100-year level.
- `cyclone_fit_unreliable` or `other_fit_unreliable` is `True` when a population has fewer than 30 storms or a shape parameter below -0.5.
- `mk_p`, `sen_slope_m_per_yr`, `pettitt_year` and `pettitt_p` summarise the trend and change-point tests of the annual maxima.

Relative depth, linear shoaling coefficient and the ratio of the 100-year Hs to the water depth (uses the depths in `config/stations.yaml`):

In [ ]:
!python scripts/depth_diagnostics.py --stats "{RESULTS_DIR}/tables/station_descriptive_stats.csv" --evt "{RESULTS_DIR}/evt/evt_summary.csv" --out "{RESULTS_DIR}/tables"

## 9. LSTM training

A GPU runtime is required (*Runtime > Change runtime type > T4 GPU*). After switching runtime type, re-run steps 1 and 2.

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

Run time per model, measured on one configuration (P1, 24 h, Huber loss, seed 0), and the resulting estimates for the three presets:

In [ ]:
!python scripts/run_lstm.py --data "{STATION_DIR}" --out "{RESULTS_DIR}/lstm_timing" --stations P1 --horizons 24 --losses huber --seeds 0
t = pd.read_csv(f'{RESULTS_DIR}/lstm_timing/lstm_metrics.csv')['minutes'].iloc[0]
print(f'one run takes {t} min')
for name, runs in {'quick (135 runs)': 135, 'core (375 runs)': 375, 'full (900 runs)': 900}.items():
    print(f'{name}: about {runs * t / 60:.1f} hours')

Presets (`--preset`): `quick` = 3 losses x 3 seeds, `core` = 5 losses x 5 seeds, `full` = 6 losses x 10 seeds, each for 5 stations x 3 lead times.
Models are trained in chunks (`--time-budget-min`); predictions and metrics are written after every run, and runs that already exist are skipped.

In [ ]:
!python scripts/run_lstm.py --data "{STATION_DIR}" --out "{RESULTS_DIR}/lstm" --preset core --time-budget-min 80

Re-run the cell until the script reports that no runs remain (`runs planned, N already finished` with N equal to the total).
After a disconnect, re-run steps 1 and 2 and then this cell.

In [ ]:
done = len(pd.read_csv(f'{RESULTS_DIR}/lstm/lstm_metrics.csv'))
print(f'{done} runs finished (core preset: 375)')

## 10. Storm evaluation, aggregated tables and figures

Can also be run on a CPU runtime and on partial results; see the appendix for working with an incomplete set of seeds.

In [ ]:
!python scripts/storm_eval.py --baselines "{RESULTS_DIR}/baselines" --lstm "{RESULTS_DIR}/lstm" --out "{RESULTS_DIR}/storms"
!python scripts/aggregate_forecast.py --baselines "{RESULTS_DIR}/baselines" --lstm "{RESULTS_DIR}/lstm" --out "{RESULTS_DIR}/tables"
!python scripts/make_figures.py --results "{RESULTS_DIR}" --out "{RESULTS_DIR}/figures"

Outputs:
- `tables/table2_forecast_skill.csv`: mean and standard deviation over seeds for every station, lead time and model, including the baselines;
- `tables/loss_pairs.csv` and `tables/loss_pair_counts.csv`: Huber loss against the other losses with paired tests;
- `tables/vs_baselines.csv`: LSTM against persistence, ridge regression and XGBoost (skill score and paired tests);
- `storms/storm_table3.csv`: skill during the named storms;
- `figures/`: data-driven figures (PDF and PNG).

## 11. Optional: altimetry match-up

Collocates the multi-mission altimeter wave heights with the station reanalysis series (0.25 degree radius, +/-1.5 h, passes defined with a 10-min gap rule) and scores the match-ups.

In [ ]:
!python scripts/altimetry_matchup.py --alt-dir "{ALT_DIR}" --stations-dir "{STATION_DIR}" --out "{RESULTS_DIR}/altimetry"
!python scripts/make_altimetry_figure.py --passes "{RESULTS_DIR}/altimetry" --out "{RESULTS_DIR}/figures"

## 12. Archive the results

Creates a compact archive of the result tables and figures (prediction files, histories and station series are excluded) and reports the storage used.

In [ ]:
shutil.copytree(REPO_DIR / 'config', OUT_DIR / 'config', dirs_exist_ok=True)    # record of the settings used
archive = PROJECT_DIR / 'results_archive.zip'
exclude = {'predictions', 'history', 'stations', 'lstm_timing'}
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in pathlib.Path(OUT_DIR).rglob('*'):
        if f.is_file() and not (exclude & set(f.parts)) and f.suffix != '.nc':
            z.write(f, f.relative_to(OUT_DIR))
print(archive, round(archive.stat().st_size / 1e6, 1), 'MB')

In [ ]:
!du -sh "{RESULTS_DIR}"/* | sort -h
!df -h /content/drive | tail -1

## Appendix: limited GPU availability and partial results

Free Colab GPU quota can run out during a long training job. Finished runs are kept on Drive, and the steps below allow the work to continue.

### A1. Check which runs are finished

In [ ]:
!python scripts/lstm_status.py --lstm "{RESULTS_DIR}/lstm" --seeds 0 1 2 3 4

The table shows, for every station, lead time and loss, how many of the requested seeds are finished. To plan for a smaller design, restrict the seeds:

In [ ]:
!python scripts/lstm_status.py --lstm "{RESULTS_DIR}/lstm" --seeds 0 1 2

### A2. Complete a smaller design first
Completing three seeds at every station before adding further seeds gives a uniform design as early as possible. The command skips runs that already exist.

In [ ]:
!python scripts/run_lstm.py --data "{STATION_DIR}" --out "{RESULTS_DIR}/lstm" --losses mse huber focal_mse focal_mse_norm target_weighted_mse --seeds 0 1 2 --time-budget-min 80

### A3. Run the remaining models in another environment
The training script runs on any machine with PyTorch and a GPU (for example a Kaggle notebook or a university cluster). Provide the repository and the
station CSV files as inputs, run `scripts/run_lstm.py` with `--out` pointing to a separate folder (for example `lstm_part2`) and with the stations or seeds still missing,
copy that folder to Drive, and merge it:

In [ ]:
part = pathlib.Path(f'{RESULTS_DIR}/lstm_part2')
main = pathlib.Path(f'{RESULTS_DIR}/lstm')
if part.exists():
    a = pd.read_csv(main / 'lstm_metrics.csv')
    b = pd.read_csv(part / 'lstm_metrics.csv')
    merged = pd.concat([a, b]).drop_duplicates(['station', 'horizon_h', 'loss', 'seed'], keep='first')
    for sub in ('predictions', 'history'):
        (main / sub).mkdir(exist_ok=True)
        for f in (part / sub).glob('*'):
            if not (main / sub / f.name).exists():
                shutil.copy(f, main / sub / f.name)
    merged.to_csv(main / 'lstm_metrics.csv', index=False)
    print(len(a), '+', len(b), '->', len(merged), 'runs')
else:
    print('no lstm_part2 folder found')

### A4. Evaluate with a common subset of seeds
When the stations have different numbers of finished seeds, restrict the evaluation to a seed set that exists everywhere. Results for stations without finished
runs are empty or based on fewer runs, so treat such tables as preliminary.

In [ ]:
SEEDS = '0 1 2'     # seed set available at every station
!python scripts/storm_eval.py --baselines "{RESULTS_DIR}/baselines" --lstm "{RESULTS_DIR}/lstm" --out "{RESULTS_DIR}/storms" --seeds {SEEDS}
!python scripts/aggregate_forecast.py --baselines "{RESULTS_DIR}/baselines" --lstm "{RESULTS_DIR}/lstm" --out "{RESULTS_DIR}/tables" --seeds {SEEDS}
!python scripts/make_figures.py --results "{RESULTS_DIR}" --out "{RESULTS_DIR}/figures" --seeds {SEEDS}